# Pipeline 2: YOLOv8-Nano + MiDaS Small (LiDAR Scale-Shift Aligned)
**Owner:** Member 2 (Emirhan) | **Branch:** `pipeline-2`

This notebook runs **only Pipeline 2** steps:
1. Clones your branch from GitHub and mounts your Google Drive.
2. Downloads the official KITTI dataset (including LiDAR) and runs `01_prepare_data.py` (preserving locked `data/splits/*.txt` if present).
3. Fine-tunes `YOLOv8-Nano` across 3 random seeds (`42, 123, 999`).
4. Backs up trained weights (`checkpoints/yolov8n_kitti_seed*.pt`) to Google Drive.
5. Verifies MiDaS Small + per-image LiDAR scale-shift alignment accuracy.

In [ ]:
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SHARED_DRIVE = '/content/drive/MyDrive/CS423_Shared_Project'
os.makedirs(f'{SHARED_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{SHARED_DRIVE}/results', exist_ok=True)
print('Google Drive backup folder ready:', SHARED_DRIVE)

In [ ]:
REPO_URL = "https://github.com/nehirvardar/cs423-assistive-navigation.git"
BRANCH = "pipeline-2"  # Your branch name

!rm -rf /content/project
!git clone -b {BRANCH} {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

In [ ]:
# Full KITTI download (including Velodyne LiDAR, required for MiDaS scale-shift alignment)
!python scripts/download_kitti.py
!python scripts/01_prepare_data.py

In [ ]:
# Fine-tune YOLOv8-Nano across 3 random seeds
!python scripts/03_train_yolo.py --model yolov8n --seeds 42 123 999 --epochs 50 --batch 16 --device 0

# Back up trained weights to Google Drive
!cp -v checkpoints/yolov8n_kitti_seed*.pt {SHARED_DRIVE}/checkpoints/

In [ ]:
# Quick verification for Pipeline 2 (first 50 test frames with MiDaS LiDAR alignment)
!python scripts/04_evaluate_all.py --split test --max-frames 50 --device cuda
!cp -r results {SHARED_DRIVE}/